# Data Science Infrastructure - Spark Structured Streaming and Kafka

This iPython notebooks shows a Spark/Kafka application for a real-world scenario (Wikipedia) and was taken from here: https://towardsdatascience.com/enabling-streaming-data-with-spark-structured-streaming-and-kafka-93ce91e5b435


## Kafka part - Producing events from Wikipedia

To simulate an application generating events for Kafka, I created this simple Python producer application that reads events from Wikipedia and sends them to the wiki-changes Kafka topic.

The message is serialized in JSON format to send to the Kafka topic. The library sseclient is required to read the source events and the kafka-python enables the Python to produce Kafka messages.

* pip install sseclient
* pip install kafka-python

In [ ]:
#!pip install sseclient

In [ ]:
## Event Producer
import json
from sseclient import SSEClient as EventSource
from kafka import KafkaProducer

# Create producer
producer = KafkaProducer(
    bootstrap_servers='kafka:9092', #Kafka server
    value_serializer=lambda v: json.dumps(v).encode('utf-8') #json serializer
    )

# Read streaming event
url = 'https://stream.wikimedia.org/v2/stream/recentchange'
try:
    for event in EventSource(url):
        if event.event == 'message':
            try:
                change = json.loads(event.data)
            except ValueError:
                pass
            else:
                #Send msg to topic wiki-changes
                producer.send('wiki-changes', change)

except KeyboardInterrupt:
    print("process interrupted")

To be observed using the command-line tool (depends on the server configuration):
* [/usr/local/kafka/bin/kafka-console-consumer.sh --bootstrap-server localhost:9092 --topic wiki-changes]
* sudo /home/kafka/kafka/bin/kafka-console-consumer.sh --bootstrap-server localhost:9092 --topic wiki-changes

In [ ]:
from kafka import KafkaConsumer
import json
import pandas as pd

from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

consumer = KafkaConsumer(
 bootstrap_servers='kafka:9092',
 value_deserializer = lambda v: json.loads(v.decode('ascii')),
 auto_offset_reset='earliest'
)

consumer.subscribe(topics='wiki-changes')


## In case you need to delete a Topic with Python

In [ ]:
from kafka.admin import KafkaAdminClient
admin_client = KafkaAdminClient(
    bootstrap_servers="kafka:9092", 
    client_id='test')


In [ ]:
admin_client.delete_topics(['wiki-changes'])